In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 재샘플링과 Order Analysis — 비정상 회전 신호 분석 + TSA

## 학습 목표
- 회전 속도가 변하는 신호에서 시간 영역 FFT 가 퍼지는 현상과 각도 기반 재샘플링 (order tracking) 을 다룬다
- Zero-padding 과 trig_rot=1 두 가지로 leakage 를 해결한다
- TSA (Time Synchronous Averaging) 로 주기적 이상치를 드러낸다
- GMF·사이드밴드 성분을 제거한 잔차(residual)·차이(difference) 신호와 포락선 스펙트럼을 분석한다
- 기어 상태 피처와 cepstrum·EEMD·웨이블릿 분석으로 정상/고장을 정량 비교한다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from utils import fft, resampling, hilbert_envelope

---

## 실습 준비 1. 신호 생성 — 회전속도가 변하는 기어 신호

- 회전 속도 $f_{rot}(t) = 1 + t$ [Hz], 누적 위상 $\phi(t) = \int_0^t f_{rot}\,d\tau$
- 기어 이빨 `n_gear = 15`개 + 축 1x 진폭 변조 → GMF(Order 15)와 사이드밴드(Order 14, 16)

$$v(t) = \left(1 + m_{AM}\sin(2\pi\phi)\right)\,\sin(n_{gear}\cdot 2\pi\phi)$$

In [ ]:
fs = 1000
T = 20
t = np.arange(1/fs, T + 1/fs, 1/fs)
n_gear = 15

f_rotation = 1 + t                       # 순시 회전 주파수 (Hz)
phase_deg = np.cumsum(f_rotation) * 360 / fs
phase_rad = 2 * np.pi * phase_deg / 360

# 축 1x 진폭 변조 → 사이드밴드 (Order 14, 16)
m_AM = 0.2
v = (1 + m_AM * np.sin(phase_rad)) * np.sin(n_gear * phase_rad)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7))

axes[0].plot(t, f_rotation, color='C0', lw=1.0)
axes[0].set_ylabel('Rotation speed (rev/s)')
axes[0].set_title('(a) Rotation speed vs time')

axes[1].plot(t, v, color='C1', lw=0.6)
axes[1].set_ylabel('Amplitude')
axes[1].set_title('(b) Vibration waveform')
axes[1].set_xlim(0, min(t[-1], 3))

f_time, A_time = fft(v, fs)
axes[2].plot(f_time, A_time, color='C3', lw=0.8)
axes[2].set_xlim(0, 500); axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_title('(c) Time-domain FFT')

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 5))
axes[0].plot(t, v, color='C0')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('v')
axes[0].set_title('Total Signal')
axes[1].plot(t, v, '.-', color='C0', markersize=3)
axes[1].set_xlim([0, 1])
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('v')
axes[1].set_title('First 1s zoom')
axes[2].plot(t, v, '.-', color='C0', markersize=3)
axes[2].set_xlim([19, 20])
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('v')
axes[2].set_title('Last 1s zoom')
fig.tight_layout(); plt.show()

---

## 실습 1. 시간 영역 FFT — 스펙트럼이 퍼지는 현상

회전 속도가 시간에 따라 변하면, 같은 기어 주파수 성분이 여러 주파수에 걸쳐 퍼져 나타난다.

In [ ]:
f, A = fft(v, fs)

plt.figure(figsize=(9, 4))
plt.plot(f, A, color='C0')
plt.xlim([0, 500])
plt.xlabel('Frequency (Hz)'); plt.ylabel('|Y|')
plt.title('Time-domain FFT')
plt.tight_layout(); plt.show()

**관찰.** 기어 메쉬 주파수 (15 × 회전 주파수) 가 15 Hz 에서 315 Hz 까지 쓸려 올라가므로, 시간 축 FFT 에는 하나의 피크 대신 15 ~ 315 Hz 에 걸친 넓고 평평한 분포 (높이 약 0.013, 최대 0.017) 가 나타난다. 이 스펙트럼에서는 메쉬 성분과 사이드밴드를 구분해 읽을 수 없다.

---

## 실습 2. 각도 기반 재샘플링 — Resampling Frequency 결정

- 전체 회전수 $N_{rot} = \phi_{\text{end}} / 360$
- 회전 1바퀴당 샘플 수 `fs_re` 를 결정 (데이터 기반 추정치를 참고, 여기서는 100으로 지정)

In [ ]:
N_rot = phase_deg[-1] / 360
N_samples = len(v)
fs_re_est = np.floor(N_samples / N_rot)

print(f'Total rotations              : {N_rot:.2f}')
print(f'Total samples                 : {N_samples}')
print(f'Samples per rotation (est.)   : {fs_re_est:.0f}')

fs_re = 100  # 회전 1바퀴당 100 샘플로 재샘플링

**관찰.** 20 초 동안 축이 약 220 회전하고 샘플은 20000 개이므로 평균은 회전당 약 90 샘플이다. 회전 주파수가 1 → 21 Hz 로 변하므로 실제로는 저속 구간에서 회전당 샘플이 훨씬 많고 (약 1000) 고속 구간에서는 적다 (약 48). 재샘플링 뒤에는 모든 회전이 똑같이 `fs_re = 100` 샘플이 된다.

---

## 실습 3. 재샘플링 직접 구현 — 원리 이해

1. 각도 축에 등간격 격자 생성 (`degree_re_delta = 360 / fs_re`)
2. 각도-시간 보간: 등간격 각도에 해당하는 시간 `t_re`
3. 시간-신호 보간: `t_re` 시점의 신호 값 `v_re`

In [ ]:
starting = phase_deg[0]
ending = phase_deg[-1]
degree_re_delta = 360 / fs_re
degree_re = np.arange(starting + degree_re_delta, ending, degree_re_delta)

fx_t_re = interp1d(phase_deg, t)
t_re = fx_t_re(degree_re)

fx_v_re = interp1d(t, v)
v_re = fx_v_re(t_re)

print(f'Number of resampling points: {len(v_re)}')
print(f'First 4 angles    : {degree_re[:4]}')
print(f'Last 4 angles    : {degree_re[-4:]}')

**관찰.** 3.6° 간격의 등각도 격자 위에 22000 개 점이 만들어진다 — 약 220 회전 × 회전당 100 점이다. 가로축이 일정한 각도 간격이 되었으므로 이 신호를 FFT 하면 가로축은 회전당 사이클 수, 곧 차수 (order) 가 된다.

---

## 실습 4. Order Analysis — 각도 축 FFT

재샘플링된 신호를 FFT 하면 주파수 축이 **Order** (회전당 주기 수) 축으로 바뀐다. 기어 이빨 수 `n_gear = 15` 이므로 **Order = 15** 에 첨예한 피크가, 좌우 **Order = 14, 16** 에 사이드밴드가 나타나야 한다.

In [ ]:
order, order_A = fft(v_re, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('Total Order Spectrum')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([13.8, 16.2])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Zoom (GMF & sidebands)')
fig.tight_layout(); plt.show()

**관찰.** 시간 축에서 넓게 퍼졌던 메쉬 성분이 차수 15 한 빈에 모여 피크가 0.85 로 선다 — 실습 1 의 최대값 0.017 의 약 49 배다. 확대하면 차수 14 · 16 에 축 1 차 진폭 변조가 만든 사이드밴드 (0.087 · 0.083) 가 분리되어 보인다.

---

## 실습 5. Trig=0 재샘플링 (utils 함수 활용)

`trig_rot=0` 은 정수 회전수를 맞추지 않는다 → 성분이 FFT bin 중심에서 벗어나 Order 15 (와 14, 16) 근처가 퍼진다.

In [ ]:
trig = 0
t_re, v_re, degree_re, fs_re = resampling(t, v, phase_deg, fs_re, trig)
N_re = len(v_re)
print(f'Resampled signal length: {N_re}, fs_re={fs_re}')

order, order_A = fft(v_re, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('Trig=0: Total Order Spectrum')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([13.8, 16.2])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Trig=0: Zoom')
fig.tight_layout(); plt.show()
order_trig0, A_trig0 = order.copy(), order_A.copy()

m = (order >= 13.8) & (order <= 16.2)
E = order_A[m] ** 2
if E.size:
    leakage = 100 * (np.sum(E) - np.sum(np.sort(E)[-3:])) / np.sum(E)
    print(f'Trig=0        — leakage outside 3 peak bins: {leakage:.1f} %')

**관찰.** `trig_rot=0` 의 재샘플링 길이는 21999 샘플로 회전당 샘플 수 100 의 정수배가 아니다. 성분이 빈 중심에서 벗어나 차수 15 피크가 0.82 로 낮아지고 주변으로 퍼진다 — 줌 구간 에너지 가운데 세 피크 빈 밖으로 새어 나간 비율이 4.9 % 이다.

---

## 실습 6. 해결방법 1 — Zero-padding

신호 뒤쪽을 0으로 채워 길이를 `fs_re`의 정수배로 맞춘다.

In [ ]:
pad_len = int(fs_re - np.mod(N_re, fs_re))
v_re_pad = np.append(v_re, np.zeros(pad_len))
print(f'Length after zero-padding: {len(v_re_pad)}')

order, order_A = fft(v_re_pad, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('After Zero-padding: Total Order Spectrum')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([13.8, 16.2])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Zero-padding: Zoom')
fig.tight_layout(); plt.show()

# leakage: 줌 창 에너지 중 3개 피크 bin 밖 비율
m = (order >= 13.8) & (order <= 16.2)
E = order_A[m] ** 2
if E.size:
    leakage = 100 * (np.sum(E) - np.sum(np.sort(E)[-3:])) / np.sum(E)
    print(f'Zero-padding  — leakage outside 3 peak bins: {leakage:.1f} %')
order_zp, A_zp = order.copy(), order_A.copy()

**관찰.** 0 을 1 개 붙여 22000 샘플 (정확히 220 회전) 로 맞추면 세 성분이 빈 중심에 놓여 피크가 다시 첨예해지고, 세 피크 빈 밖으로 새는 비율이 0.9 % 로 줄어든다.

---

## 실습 7. 해결방법 2 — Trig=1 (정수 회전수만 유지)

`utils.resampling(..., trig_rot=1)` 은 정수 회전수를 넘는 뒤쪽 샘플을 잘라버린다. → 신호 길이가 정확히 n 바퀴에 해당한다.

In [ ]:
trig = 1
t_re, v_re, degree_re, fs_re = resampling(t, v, phase_deg, fs_re, trig)
N_re = len(v_re)
print(f'Trig=1 resampled length: {N_re}  ({N_re % fs_re == 0} → integer rotation multiple)')

order, order_A = fft(v_re, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('Trig=1 (integer rotations): Total Order Spectrum')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([13.8, 16.2])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Trig=1: Zoom')
fig.tight_layout(); plt.show()

m = (order >= 13.8) & (order <= 16.2)
E = order_A[m] ** 2
if E.size:
    leakage = 100 * (np.sum(E) - np.sum(np.sort(E)[-3:])) / np.sum(E)
    print(f'Trig=1        — leakage outside 3 peak bins: {leakage:.1f} %')
order_trig1, A_trig1 = order.copy(), order_A.copy()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
cases = [
    ('Trig=0',       order_trig0, A_trig0, 'C0'),
    ('Zero-padding', order_zp,    A_zp,    'C1'),
    ('Trig=1',       order_trig1, A_trig1, 'C2'),
]
for ax, (lbl, o, a, color) in zip(axes, cases):
    ax.plot(o, a, '-o', color=color, markersize=4, lw=1.0)
    ax.set_xlim(13.8, 16.2); ax.set_xlabel('Order')
    ax.set_title(lbl)
    for o_line in [14, 15, 16]:
        ax.axvline(o_line, color='gray', ls=':', lw=0.8)
axes[0].set_ylabel('Amplitude')
fig.suptitle('Order 14-16 Zoom — Three Resampling Strategies')
plt.tight_layout(); plt.show()

**관찰.** `trig_rot=1` 은 정수 회전을 넘는 뒤쪽을 잘라 21900 샘플 (219 회전) 로 맞추며, 새는 비율은 제로패딩과 같은 0.9 % 이다. 세 방법을 나란히 놓으면 Trig=0 만 차수 15 주변에 스커트가 남고, 제로패딩과 Trig=1 은 차수 14 · 15 · 16 세 빈에 에너지가 모인다 — 신호 길이를 정수 회전으로 맞추는 것이 차수 스펙트럼의 누설을 막는 핵심이다.

---

## 실습 준비 2. 주기적 이상치 추가 (정상 vs 고장)

각 회전마다 동일한 위치(40번째 샘플)에 진폭 0.8 펄스를 더해 기어 1바퀴당 1회 이상치가 나타나는 고장 신호를 만든다.

In [ ]:
v_re_f = np.copy(v_re_pad)                # fault
v_re_n = np.copy(v_re_pad)                # normal
idx_fault = np.arange(40, len(v_re_f), fs_re)
v_re_f[idx_fault] = v_re_f[idx_fault] + 0.8

fig, axes = plt.subplots(2, 1, figsize=(9, 5))
axes[0].plot(v_re_f, color='C3', label='Fault')
axes[0].plot(v_re_n, color='C0', alpha=0.5, label='Normal')
axes[0].legend(); axes[0].set_title('Impact added')
axes[1].plot(v_re_f, color='C3', label='Fault')
axes[1].plot(v_re_n, color='C0', alpha=0.5, label='Normal')
axes[1].set_xlim([0, 300]); axes[1].legend()
axes[1].set_title('Zoom')
axes[1].set_xlabel('Samples'); axes[1].set_ylabel('v')
fig.tight_layout(); plt.show()

---

## 실습 준비 3. 노이즈 추가 — 이상치가 가려짐

In [ ]:
np.random.seed(1000)
noise_f = np.random.randn(len(v_re_f)) / 2      # 표준편차 0.5 가우시안 노이즈
noise_n = np.random.randn(len(v_re_n)) / 2
v_re_f = v_re_f + noise_f
v_re_n = v_re_n + noise_n

plt.figure(figsize=(9, 4))
plt.plot(v_re_f, color='C3', label='Fault')
plt.plot(v_re_n, color='C0', alpha=0.5, label='Normal')
plt.xlim([0, 1000])
plt.xlabel('Samples'); plt.ylabel('v'); plt.legend()
plt.title('Noise added')
plt.tight_layout(); plt.show()

---

## 실습 8. TSA — 신호를 회전주기로 잘라서 평균

재샘플링된 신호는 한 회전이 정확히 `fs_re` 샘플이다. 회전마다 잘라 쌓고 세로로 평균하면 회전 동기 성분(기어 메쉬, 매 회전 같은 각도의 이상치)은 그대로 남고 비동기 노이즈는 평균 횟수 $N$ 에 대해 $1/\sqrt{N}$ 로 줄어든다.

1. **1 회전 단위 TSA** — `fs_re` 샘플씩 잘라 전체 회전을 평균하고 가로축을 회전 각도로 그린다. 노이즈만 같은 방식으로 평균해 RMS 가 $1/\sqrt{N}$ 을 따르는지 확인한다.
2. **5 회전 묶음 TSA** — `5 * fs_re` 샘플씩 잘라 평균한다 (길이를 `seg_len` 배수로 절단하는 안전장치 포함). 평균 횟수는 1/5 로 줄지만 한 묶음 안에 회전 사이의 반복 구조가 남으므로, 실습 9 이후의 분석 (포락선 스펙트럼의 1x 배수, cepstrum 의 1 rev 배수 등) 은 이 신호를 쓴다.

In [ ]:
# (1) 1 회전 단위 TSA — 회전 각도에 맞춰 전체 회전을 평균
n_rot = len(v_re_f) // fs_re
angle = np.arange(fs_re) / fs_re * 360           # 회전 안의 각도 (deg)
seg_f = v_re_f[:n_rot * fs_re].reshape(n_rot, fs_re)
seg_n = v_re_n[:n_rot * fs_re].reshape(n_rot, fs_re)
tsa1_f = seg_f.mean(axis=0)
tsa1_n = seg_n.mean(axis=0)

# 노이즈만 같은 방식으로 평균 -> RMS 가 1/sqrt(N) 로 준다
seg_noise = noise_f[:n_rot * fs_re].reshape(n_rot, fs_re)
rms_1 = np.sqrt(np.mean(seg_noise[0] ** 2))
rms_N = np.sqrt(np.mean(seg_noise.mean(axis=0) ** 2))
print(f'Revolutions averaged: N = {n_rot}')
print(f'Noise RMS: {rms_1:.3f} (N=1) -> {rms_N:.4f} (N={n_rot}), '
      f'reduction {rms_1 / rms_N:.1f}x  (sqrt(N) = {np.sqrt(n_rot):.1f})')

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True, sharey=True)
axes[0].plot(angle, seg_f[0], color='C3')
axes[0].set_title('One revolution, raw (Fault)')
axes[0].set_ylabel('v')
axes[1].plot(angle, tsa1_n, color='C0', label='Normal TSA')
axes[1].plot(angle, tsa1_f, color='C3', label='Fault TSA')
axes[1].set_title(f'TSA over {n_rot} revolutions')
axes[1].set_xlabel('Angle within one revolution (deg)'); axes[1].set_ylabel('v')
axes[1].set_xlim(0, 360); axes[1].legend()
fig.tight_layout(); plt.show()

In [ ]:
# (2) 5 회전 묶음 TSA — 실습 9 이후에서 사용
n_rot_TSA = 5
seg_len = n_rot_TSA * fs_re

# 안전장치: v_re 길이를 seg_len 배수로 맞춤
n_full = (len(v_re_f) // seg_len) * seg_len
v_re_f = v_re_f[:n_full]
v_re_n = v_re_n[:n_full]

v_reshape_f = v_re_f.reshape(seg_len, -1, order='F')
v_TSA_f = np.mean(v_reshape_f, axis=1)

v_reshape_n = v_re_n.reshape(seg_len, -1, order='F')
v_TSA_n = np.mean(v_reshape_n, axis=1)

print(f'segment shape: {v_reshape_f.shape} (seg_len={seg_len}, n_segments={v_reshape_f.shape[1]})')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6))
axes[0].plot(v_reshape_f, color='C0', alpha=0.15)
axes[0].plot(v_TSA_f, '--', color='C3', linewidth=2.5, label='TSA Mean')
axes[0].legend(); axes[0].set_title('Segments + TSA mean (Fault)')
axes[1].plot(v_TSA_f, color='C3', label='Fault TSA')
axes[1].plot(v_TSA_n, color='C0', label='Normal TSA')
axes[1].set_xlabel('Samples (5 rotations)'); axes[1].legend()
axes[1].set_title('Fault vs Normal TSA')
fig.tight_layout(); plt.show()

**관찰.**
- 1 회전 단위: 한 회전만 보면 (위) 진폭 0.8 의 이상치가 표준편차 0.5 의 노이즈에 묻혀 어느 각도인지 고를 수 없다. 220 회전을 평균하면 (아래) 회전당 15 주기의 메쉬 파형이 드러나고, Fault TSA 에만 144° (40 번째 샘플) 자리에 이상치가 남는다.
- 노이즈 RMS 는 0.527 → 0.0326 으로 16.2 배 줄었고, 0.0326 은 이론값 $0.5/\sqrt{220} \approx 0.0337$ 과 거의 같다. 줄어든 비율이 $\sqrt{220} \approx 14.8$ 보다 조금 큰 것은 첫 회전의 노이즈 RMS (0.527) 가 이론값 0.5 보다 컸기 때문이다.
- 5 회전 묶음: 44 개 묶음을 평균한 500 샘플 TSA 에서는 이상치가 100 샘플 (1 회전) 간격으로 다섯 번 나타난다. 평균 횟수가 1/5 이라 노이즈는 덜 줄지만 ($\sqrt{44} \approx 6.6$ 배), 회전 사이의 반복 구조를 다음 실습들에서 쓸 수 있다.

---

## 실습 9. TSA 신호의 주파수 분석

기어 정규 성분은 GMF(Order 15)·고조파(30, 45)와 사이드밴드(14, 16)로, 회전당 1회 반복되는 이상치는 shaft order 1 의 배수(정수 order) 성분으로 나타난다.

In [ ]:
f_raw_n, A_raw_n = fft(v_re_n, fs_re)
f_raw_f, A_raw_f = fft(v_re_f, fs_re)
f_n, A_n = fft(v_TSA_n, fs_re)
f_f, A_f = fft(v_TSA_f, fs_re)

specs = [
    ('Raw', f_raw_n, A_raw_n, f_raw_f, A_raw_f),
    ('TSA', f_n, A_n, f_f, A_f),
]
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for i, (name, fo_n, Ao_n, fo_f, Ao_f) in enumerate(specs):
    for j in range(2):
        axes[i, j].plot(fo_f, Ao_f, color='C3', label='Fault')
        axes[i, j].plot(fo_n, Ao_n, color='C0', alpha=0.7, label='Normal')
        axes[i, j].set_ylabel('|Y|'); axes[i, j].legend()
    axes[i, 0].set_title(f'{name} Spectrum')
    axes[i, 1].set_title(f'{name} Spectrum (Zoom)')
    axes[i, 1].set_ylim([0, 0.05])
axes[1, 0].set_xlabel('Order'); axes[1, 1].set_xlabel('Order')
fig.tight_layout(); plt.show()

**관찰.** 원 신호 (위) 의 확대 스펙트럼은 빈이 1 만 개 남짓이라 잡음 바닥이 0.015 까지 빽빽한 띠로 보이고, 고장의 정수 차수 선 (중앙값 약 0.017) 은 일부만 띠 위로 솟는다. TSA (아래) 의 스펙트럼은 원 신호 스펙트럼에서 44 번째 빈마다 하나씩 (5 회전 묶음에 맞는 0.2 차수 간격의 빈) 만 남긴 것과 같다. 빈 하나의 잡음 높이는 그대로 (정상 바닥 중앙값 약 0.006) 이지만 회전과 무관한 빈이 사라져, 고장에만 정수 차수마다 0.01 ~ 0.025 의 선이 줄지어 나타나는 것이 또렷해진다 — 회전당 1 회 반복되는 이상치가 차수 1 의 배수 성분으로 나타난 것이다.

---

## 실습 10. 잔차·차이 신호 — Order 성분 제거와 포락선 스펙트럼

TSA 신호에서 정규(regular) 성분 — GMF 고조파(15·30·45)와 사이드밴드(±1) — 을 제거한 신호를 본다. TSA 는 성분이 정확히 bin 중심에 있으므로 해당 bin 을 0 으로 만드는 방식(`remove_orders`)으로 제거한다.

| 파생 신호 | 구성 | 남는 것 |
|---|---|---|
| 잔차 $v_{RES}$ (residual) | TSA − GMF 고조파 | 사이드밴드 + 임펄스 + 노이즈 |
| 차이 $v_{DIF}$ (difference) | TSA − (GMF 고조파 + 사이드밴드) | 임펄스 + 노이즈 |

In [ ]:
# GMF 고조파와 1차 사이드밴드 정의 (n_gear=15, Nyquist = fs_re/2 = 50)
gmf_harmonics = [15, 30, 45]
sidebands = [14, 16, 29, 31, 44, 46]      # 각 GMF 고조파 ± 1 (shaft order = 1)

def remove_orders(v, fs_re, orders, half_width=0.3):
    """FFT 에서 지정 order 근방 bin 을 0으로 만들고 역변환 (order-domain notch)."""
    V = np.fft.rfft(v)
    order_axis = np.fft.rfftfreq(len(v), d=1/fs_re)
    for o in orders:
        V[np.abs(order_axis - o) <= half_width] = 0
    return np.fft.irfft(V, n=len(v))

# residual: GMF 고조파만 제거
v_res_n = remove_orders(v_TSA_n, fs_re, gmf_harmonics)
v_res_f = remove_orders(v_TSA_f, fs_re, gmf_harmonics)

# difference: 사이드밴드까지 제거
v_dif_n = remove_orders(v_TSA_n, fs_re, gmf_harmonics + sidebands)
v_dif_f = remove_orders(v_TSA_f, fs_re, gmf_harmonics + sidebands)

# regular: 제거된 정규 성분
v_reg_n = v_TSA_n - v_dif_n
v_reg_f = v_TSA_f - v_dif_f

signals = [
    ('TSA', v_TSA_n, v_TSA_f),
    ('Residual', v_res_n, v_res_f),
    ('Difference', v_dif_n, v_dif_f),
]
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
for i, (name, sn, sf) in enumerate(signals):
    axes[i, 0].plot(sf, color='C3', label='Fault')
    axes[i, 0].plot(sn, color='C0', alpha=0.7, label='Normal')
    axes[i, 0].set_ylabel('Amplitude'); axes[i, 0].legend()
    axes[i, 0].set_title(f'{name} — time')

    fo_n, Ao_n = fft(sn, fs_re)
    fo_f, Ao_f = fft(sf, fs_re)
    axes[i, 1].plot(fo_f, Ao_f, color='C3', label='Fault')
    axes[i, 1].plot(fo_n, Ao_n, color='C0', alpha=0.7, label='Normal')
    axes[i, 1].set_ylim([0, 0.05])
    axes[i, 1].set_ylabel('|Y|'); axes[i, 1].legend()
    axes[i, 1].set_title(f'{name} — order spectrum')
axes[2, 0].set_xlabel('Samples (5 rotations)')
axes[2, 1].set_xlabel('Order')
fig.tight_layout(); plt.show()

In [ ]:
# 포락선 스펙트럼: 힐베르트 포락선 → DC 제거 → FFT
env_signals = [('Raw', v_re_n, v_re_f)] + signals

fig, axes = plt.subplots(4, 1, figsize=(9, 10), sharex=True)
for ax, (name, sn, sf) in zip(axes, env_signals):
    en = hilbert_envelope(sn)
    ef = hilbert_envelope(sf)
    fe_n, Ae_n = fft(en - np.mean(en), fs_re)
    fe_f, Ae_f = fft(ef - np.mean(ef), fs_re)

    for o in range(1, 21):
        ax.axvline(o, color='gray', ls='--', lw=0.6, alpha=0.5)
    ax.plot(fe_f, Ae_f, color='C3', label='Fault')
    ax.plot(fe_n, Ae_n, color='C0', alpha=0.7, label='Normal')
    ax.set_xlim([0, 20])
    ax.set_ylabel('|Y|'); ax.legend()
    ax.set_title(f'{name} — envelope spectrum')
axes[-1].set_xlabel('Order (dashed: 1x harmonics)')
fig.tight_layout(); plt.show()

**관찰.**
- 시간 영역: TSA 에서는 메쉬 파형 위에 이상치가 실려 있지만, 잔차 (GMF 고조파 제거) 와 차이 (사이드밴드까지 제거) 신호에서는 메쉬가 사라져 100 샘플 간격의 임펄스가 고장 신호에서만 두드러진다. 정상 잔차에는 사이드밴드 (차수 14 · 16) 가 남아 진폭 약 ±0.25 의 변조 파형이 보인다.
- 포락선 스펙트럼: 원 신호와 TSA 는 정상·고장 모두 차수 1 (축 1 차 진폭 변조) 이 가장 커서 판을 지배한다. 고장 쪽에는 차수 2 ~ 20 의 작은 빗살이 이미 서 있지만 (TSA 에서 대부분 0.02 ~ 0.04, 차수 1 의 10 ~ 25 %) 차수 1 에 눌려 눈에 덜 띈다. 잔차에서는 정상이 차수 2 (사이드밴드 14 · 16 의 맥놀이) 를 보이고, 차이 신호에서는 정상은 평평한데 고장에만 1x 배수 (차수 1, 2, 3, …) 의 빗살이 나타난다 — 임펄스가 회전당 1 회 반복된다는 증거다.

---

## 실습 11. 기어 상태 피처(Condition Indicator)

TSA·잔차·차이 신호에서 스칼라 피처를 계산해 정상/고장을 수치로 비교한다.

| 기반 신호 | 피처 |
|---|---|
| $v_{TSA}$ | FM0, SER |
| $v_{RES}$ | NA4, NA4* |
| $v_{DIF}$ | FM4, M6A, M8A, ER |

- FM0 = peak-to-peak($v_{TSA}$) $/\ \sum_k A_{GMF_k}$
- SER = $\sum A_{sideband} \,/\, A_{GMF_1}$
- NA4 = $N \sum (v_{RES}-\bar{v}_{RES})^4 \,/\, \left(\overline{\sigma^2_{run}}\right)^2$ (분산은 기록 누적 평균), NA4* 는 분모를 정상 상태 분산으로 고정
- FM4 / M6A / M8A = $v_{DIF}$ 의 정규화 4/6/8차 중심 모멘트 (가우시안 기준 3, 15, 105)
- ER = RMS($v_{DIF}$) / RMS(정규 성분)

`gear_features` 함수 하나로 정상·고장에 동일 적용한다 (`var_ref` = 정상 잔차 분산, NA4 reset 은 부하 정보가 필요해 제외).

In [ ]:
def moment_ratio(x, k):
    """정규화 k차 중심 모멘트 = N^(k/2-1) * sum(x0^k) / sum(x0^2)^(k/2)  (k=4 → 첨도)"""
    x0 = x - np.mean(x)
    N = len(x0)
    return N ** (k // 2 - 1) * np.sum(x0 ** k) / np.sum(x0 ** 2) ** (k // 2)

def gear_features(v_TSA, v_res, v_dif, v_reg, fs_re, var_ref=None):
    """한 상태(정상 또는 고장)의 기어 상태 피처 8종 계산.

    var_ref : NA4/NA4* 분산 정규화 기준 (정상 상태 잔차 분산).
              None 이면 자기 자신의 분산 사용 (= 첫 번째 기록인 정상 상태).
    """
    # TSA 기반: FM0, SER
    f_ord, A_ord = fft(v_TSA, fs_re)
    def amp(o):
        return A_ord[np.argmin(np.abs(f_ord - o))]
    FM0 = np.ptp(v_TSA) / sum(amp(o) for o in gmf_harmonics)
    SER = (amp(14) + amp(16)) / amp(15)     # 기본 GMF(15) 의 1차 사이드밴드

    # 잔차 기반: NA4, NA4*
    var_res = np.var(v_res)
    m4 = np.mean((v_res - np.mean(v_res)) ** 4)
    if var_ref is None:
        var_ref = var_res
    NA4 = m4 / np.mean([var_ref, var_res]) ** 2   # 기록들의 평균 분산으로 정규화
    NA4s = m4 / var_ref ** 2                      # 정상 상태 분산으로 고정 정규화

    # 차이 신호 기반: FM4, M6A, M8A, ER
    FM4 = moment_ratio(v_dif, 4)
    M6A = moment_ratio(v_dif, 6)
    M8A = moment_ratio(v_dif, 8)
    ER = np.sqrt(np.mean(v_dif ** 2)) / np.sqrt(np.mean(v_reg ** 2))

    names = ['FM0', 'SER', 'NA4', 'NA4*', 'FM4', 'M6A', 'M8A', 'ER']
    return np.array([FM0, SER, NA4, NA4s, FM4, M6A, M8A, ER]), names

vals_n, names = gear_features(v_TSA_n, v_res_n, v_dif_n, v_reg_n, fs_re)
vals_f, _ = gear_features(v_TSA_f, v_res_f, v_dif_f, v_reg_f, fs_re, var_ref=np.var(v_res_n))

print(f'{"Feature":10s}{"Normal":>12s}{"Fault":>12s}{"Fault/Normal":>14s}')
for name, vn, vf in zip(names, vals_n, vals_f):
    print(f'{name:10s}{vn:12.3f}{vf:12.3f}{vf/vn:14.2f}')

x = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(x - 0.2, vals_n, width=0.4, color='C0', label='Normal')
axes[0].bar(x + 0.2, vals_f, width=0.4, color='C3', label='Fault')
axes[0].set_xticks(x); axes[0].set_xticklabels(names)
axes[0].set_ylabel('Feature value'); axes[0].legend()
axes[0].set_title('Condition Indicators')
axes[1].bar(x, vals_f / vals_n, color='C2')
axes[1].axhline(1, color='gray', ls=':')
axes[1].set_xticks(x); axes[1].set_xticklabels(names)
axes[1].set_ylabel('Fault / Normal ratio')
axes[1].set_title('Relative Change')
fig.tight_layout(); plt.show()

**관찰.** TSA 기반 FM0 (1.22 배) 와 SER (0.99 배) 는 거의 변하지 않는다 — 이상치가 메쉬·사이드밴드 진폭을 바꾸지 않기 때문이다. 잔차·차이 신호 기반 지표는 크게 반응한다: NA4 7.3 배, NA4* 11.5 배, FM4 6.9 배이고, 고차 모멘트일수록 M6A 60 배, M8A 417 배로 커진다. 정상의 FM4 (2.8) · M6A (13.7) · M8A (90) 는 가우시안 기준값 3 · 15 · 105 에 가깝다. ER 은 1.5 배로 완만하다.

---

## 실습 12. Cepstrum — Raw vs TSA vs DIFF

Cepstrum 은 log 스펙트럼의 역 FFT: $c(q) = \mathcal{F}^{-1}\{\log|X|\}$. 스펙트럼의 등간격 구조(고조파·사이드밴드 family)가 quefrency 피크(rahmonic)로 모인다 — 회전당 1회 임펄스는 1 rev 배수 rahmonic. Spectrum → Envelope Spectrum → Cepstrum 을 3×3 으로 비교한다.

In [ ]:
from scipy import stats

def real_cepstrum(x, fs):
    """Real cepstrum: IFFT(log|FFT|). quefrency 단위 = 회전수(rev)."""
    X = np.abs(np.fft.rfft(x))
    X = np.maximum(X, 0.1 * np.median(X))   # 제거된 bin 보호
    c = np.fft.irfft(np.log(X))
    q = np.arange(len(c)) / fs
    return q, c

cep_signals = [
    ('Raw', v_re_n, v_re_f),
    ('TSA', v_TSA_n, v_TSA_f),
    ('DIFF', v_dif_n, v_dif_f),
]
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for j, (name, sn, sf) in enumerate(cep_signals):
    # 1행: Spectrum
    ax = axes[0, j]
    fo_n, Ao_n = fft(sn, fs_re)
    fo_f, Ao_f = fft(sf, fs_re)
    ax.plot(fo_f, Ao_f, color='C3', label='Fault')
    ax.plot(fo_n, Ao_n, color='C0', alpha=0.7, label='Normal')
    ax.set_ylim([0, 0.05])
    ax.set_xlabel('Order'); ax.set_ylabel('|Y|'); ax.legend()
    ax.set_title(f'{name} — spectrum')

    # 2행: Envelope Spectrum
    ax = axes[1, j]
    en = hilbert_envelope(sn)
    ef = hilbert_envelope(sf)
    fe_n, Ae_n = fft(en - np.mean(en), fs_re)
    fe_f, Ae_f = fft(ef - np.mean(ef), fs_re)
    for o in range(1, 21):
        ax.axvline(o, color='gray', ls='--', lw=0.6, alpha=0.5)
    ax.plot(fe_f, Ae_f, color='C3', label='Fault')
    ax.plot(fe_n, Ae_n, color='C0', alpha=0.7, label='Normal')
    ax.set_xlim([0, 20])
    ax.set_xlabel('Order'); ax.set_ylabel('|Y|'); ax.legend()
    ax.set_title(f'{name} — envelope spectrum')

    # 3행: Cepstrum
    ax = axes[2, j]
    qn, cn = real_cepstrum(sn, fs_re)
    qf, cf = real_cepstrum(sf, fs_re)
    for r in range(1, 5):
        ax.axvline(r, color='gray', ls='--', lw=0.6, alpha=0.5)
    mn = (qn >= 0.1) & (qn <= 4)
    mf = (qf >= 0.1) & (qf <= 4)
    ax.plot(qf[mf], np.abs(cf[mf]), color='C3', label='Fault')
    ax.plot(qn[mn], np.abs(cn[mn]), color='C0', alpha=0.7, label='Normal')
    ax.set_xlabel('Quefrency (rev)'); ax.set_ylabel('|Cepstrum|'); ax.legend()
    ax.set_title(f'{name} — cepstrum')
fig.tight_layout(); plt.show()

**관찰.**
- 원 신호 (왼쪽 열): 스펙트럼·포락선 스펙트럼 모두 노이즈가 지배한다. 셉스트럼에서는 고장의 3 rev 자리가 판에서 가장 높지만 (약 0.016) 정상의 요동 (최대 약 0.014) 과 크게 다르지 않고 1 rev 는 정상보다 낮아, rahmonic 열로 읽을 수 없다.
- TSA (가운데 열): 셉스트럼에서 고장 신호에만 1 · 2 · 3 · 4 rev 에 0.23 ~ 0.28 의 rahmonic 이 선다 — 정수 차수마다 늘어선 선 (회전당 1 회 이상치) 이 셉스트럼에서는 회전 주기의 배수 몇 점으로 모인다.
- 차이 신호 (오른쪽 열): 포락선 스펙트럼의 1x 빗살이 가장 깨끗하다. 셉스트럼에도 고장 쪽 2 · 3 rev 에 rahmonic (약 0.21) 이 보이지만 1 · 4 rev 는 뚜렷하지 않고 낮은 quefrency 쪽 값이 커서, rahmonic 판독은 TSA 쪽이 더 분명하다.

---

## 실습 13. EEMD 분해 — Raw vs TSA vs DIFF

EMD 는 신호를 IMF(고주파→저주파 순)로 분해하고, EEMD 는 백색잡음 앙상블로 모드 혼합을 완화한다. 임펄스가 어느 IMF 에 나타나는지 IMF 별 첨도로 비교한다. (`PyEMD` 없으면 셀이 자동 설치)

In [ ]:
try:
    from PyEMD import EEMD
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'EMD-signal'], check=True)
    from PyEMD import EEMD

eemd = EEMD(trials=20, parallel=False)
eemd.noise_seed(42)

raw_n_seg = v_re_n[:seg_len]                 # 앞 5회전 구간
raw_f_seg = v_re_f[:seg_len]
imfs_raw_n = eemd.eemd(raw_n_seg)
imfs_raw_f = eemd.eemd(raw_f_seg)
imfs_tsa_n = eemd.eemd(v_TSA_n)
imfs_tsa_f = eemd.eemd(v_TSA_f)
imfs_dif_n = eemd.eemd(v_dif_n)
imfs_dif_f = eemd.eemd(v_dif_f)

idx_pulse = np.arange(40, seg_len, fs_re)    # 임펄스 위치

fig, axes = plt.subplots(5, 3, figsize=(18, 11), sharex=True)
cols = [('Raw', raw_n_seg, raw_f_seg, imfs_raw_n, imfs_raw_f),
        ('TSA', v_TSA_n, v_TSA_f, imfs_tsa_n, imfs_tsa_f),
        ('DIFF', v_dif_n, v_dif_f, imfs_dif_n, imfs_dif_f)]
for j, (name, sn, sf, imfs_n, imfs_f) in enumerate(cols):
    ax = axes[0, j]
    for p in idx_pulse:
        ax.axvline(p, color='gray', ls='--', lw=0.6, alpha=0.5)
    ax.plot(sf, color='C3', lw=0.8, label='Fault')
    ax.plot(sn, color='C0', alpha=0.6, lw=0.8, label='Normal')
    ax.set_ylabel('Amplitude'); ax.legend()
    ax.set_title(f'{name} — signal')
    for k in range(4):
        ax = axes[k + 1, j]
        for p in idx_pulse:
            ax.axvline(p, color='gray', ls='--', lw=0.6, alpha=0.5)
        ax.plot(imfs_f[k], color='C3', lw=0.8, label='Fault')
        ax.plot(imfs_n[k], color='C0', alpha=0.6, lw=0.8, label='Normal')
        ax.set_ylabel('Amplitude')
        ax.set_title(f'{name} — IMF{k+1}')
for j in range(3):
    axes[-1, j].set_xlabel('Samples (dashed: impulse positions)')
fig.tight_layout(); plt.show()

print(f'{"IMF":6s}{"Raw-N":>8s}{"Raw-F":>8s}{"TSA-N":>8s}{"TSA-F":>8s}{"DIF-N":>8s}{"DIF-F":>8s}   (kurtosis)')
for k in range(4):
    row = [stats.kurtosis(imfs[k], fisher=False)
           for imfs in [imfs_raw_n, imfs_raw_f, imfs_tsa_n, imfs_tsa_f, imfs_dif_n, imfs_dif_f]]
    print(f'IMF{k+1:<3d}' + ''.join(f'{v:8.2f}' for v in row))

**관찰.** 원 신호에서는 노이즈가 커서 어느 IMF 에도 임펄스 위치 (점선) 에 맞는 구조가 없고, 고장 IMF 의 첨도 (2.1 ~ 3.2) 가 정상보다 크지도 않다. TSA 에서는 IMF1 이 메쉬 파형 (첨도 약 1.7, 사인파에 가까움) 을 가져가고 IMF2 · IMF3 에 임펄스 위치의 진동이 남는다 (고장 첨도 3.7 · 3.5). 차이 신호에서는 메쉬가 이미 제거되어 IMF1 부터 다섯 임펄스가 점선 위치에 나타나고, 고장 첨도가 IMF1 ~ 3 에서 5.0 ~ 6.7 로 정상 (2.2 ~ 3.3) 의 약 두 배다.

---

## 실습 14. 웨이블릿 분해 — Raw vs TSA vs DIFF

DWT(`pywt.wavedec`, db4, level 3)는 신호를 반씩 좁아지는 대역으로 나눈다:

| 성분 | 대역 (Order) |
|---|---|
| cD1 | 25 ~ 50 |
| cD2 | 12.5 ~ 25 |
| cD3 | 6.25 ~ 12.5 |
| cA3 | 0 ~ 6.25 |

계수는 데시메이션으로 길이가 달라 샘플 축(0~500)으로 환산해 그린다. detail 패널의 수평 점선은 다음 실습의 denoising threshold.

In [ ]:
import pywt

def wavelet_threshold(coeffs, N):
    """레벨별 universal threshold: sigma_j = MAD(cD_j)/0.6745, lambda_j = sigma_j*sqrt(2 ln N)."""
    return [np.median(np.abs(c)) / 0.6745 * np.sqrt(2 * np.log(N)) for c in coeffs[1:]]

def coef_positions(n, k, wavelet='db4', level=3):
    """wavedec 결과 k 번 배열(0: cA3, 1: cD3, 2: cD2, 3: cD1)의 계수 번호 -> 길이 n 신호의 샘플 위치.

    가운데 샘플에 임펄스를 넣고, 그 계수 에너지의 무게중심이 임펄스 자리에 오도록 번호를 옮긴다.
    번호를 0 ~ n 에 고르게 펴기만 하면 필터 지연과 경계 확장이 빠져 계수가 임펄스에서 어긋나 그려진다."""
    step = 2 ** (level if k == 0 else level + 1 - k)   # 계수 한 칸 = 샘플 몇 개 (cA3·cD3 8, cD2 4, cD1 2)
    d = np.zeros(n)
    d[n // 2] = 1.0
    e = pywt.wavedec(d, wavelet, level=level)[k] ** 2
    i0 = np.sum(np.arange(len(e)) * e) / np.sum(e)      # 임펄스가 놓인 계수 번호 (소수)
    return n // 2 + (np.arange(len(e)) - i0) * step

wav_labels = ['cA3', 'cD3', 'cD2', 'cD1']
coeffs_raw_n = pywt.wavedec(raw_n_seg, 'db4', level=3)
coeffs_raw_f = pywt.wavedec(raw_f_seg, 'db4', level=3)
coeffs_tsa_n = pywt.wavedec(v_TSA_n, 'db4', level=3)
coeffs_tsa_f = pywt.wavedec(v_TSA_f, 'db4', level=3)
coeffs_dif_n = pywt.wavedec(v_dif_n, 'db4', level=3)
coeffs_dif_f = pywt.wavedec(v_dif_f, 'db4', level=3)

fig, axes = plt.subplots(5, 3, figsize=(18, 11), sharex=True)
cols = [('Raw', raw_n_seg, raw_f_seg, coeffs_raw_n, coeffs_raw_f),
        ('TSA', v_TSA_n, v_TSA_f, coeffs_tsa_n, coeffs_tsa_f),
        ('DIFF', v_dif_n, v_dif_f, coeffs_dif_n, coeffs_dif_f)]
for j, (name, sn, sf, cs_n, cs_f) in enumerate(cols):
    ax = axes[0, j]
    for p in idx_pulse:
        ax.axvline(p, color='gray', ls='--', lw=0.6, alpha=0.5)
    ax.plot(sf, color='C3', lw=0.8, label='Fault')
    ax.plot(sn, color='C0', alpha=0.6, lw=0.8, label='Normal')
    ax.set_ylabel('Amplitude'); ax.legend()
    ax.set_title(f'{name} — signal')

    thr_f = wavelet_threshold(cs_f, len(sf))
    for k in range(4):
        ax = axes[k + 1, j]
        x_k = coef_positions(len(sf), k)              # 계수 번호 -> 샘플 위치
        for p in idx_pulse:
            ax.axvline(p, color='gray', ls='--', lw=0.6, alpha=0.5)
        if k >= 1:                                    # detail 대역: denoising threshold 표시
            ax.axhline(thr_f[k - 1], color='C2', ls=':', lw=1.0)
            ax.axhline(-thr_f[k - 1], color='C2', ls=':', lw=1.0)
        ax.plot(x_k, cs_f[k], color='C3', lw=0.8, label='Fault')
        ax.plot(x_k, cs_n[k], color='C0', alpha=0.6, lw=0.8, label='Normal')
        ax.set_ylabel('Coefficient')
        ax.set_title(f'{name} — {wav_labels[k]}')
for j in range(3):
    axes[-1, j].set_xlabel('Samples (dashed: impulses, dotted: threshold)')
fig.tight_layout(); plt.show()

print(f'{"Band":6s}{"Raw-N":>8s}{"Raw-F":>8s}{"TSA-N":>8s}{"TSA-F":>8s}{"DIF-N":>8s}{"DIF-F":>8s}   (kurtosis)')
for k, lbl in enumerate(wav_labels):
    row = [stats.kurtosis(cs[k], fisher=False)
           for cs in [coeffs_raw_n, coeffs_raw_f, coeffs_tsa_n, coeffs_tsa_f, coeffs_dif_n, coeffs_dif_f]]
    print(f'{lbl:<6s}' + ''.join(f'{v:8.2f}' for v in row))

**관찰.** 메쉬 성분 (차수 15) 은 cD2 (12.5 ~ 25) 대역에 있어 TSA 의 cD2 · cD3 은 정상과 고장이 거의 같은 사인 파형이다. 임펄스는 넓은 대역의 cD1 · cD2 에 짧은 큰 계수로 나타난다 — 차이 신호의 cD1 에서는 다섯 임펄스가 모두 임계값 (점선) 을 넘고 고장 첨도가 10.7 (정상 3.2) 이다. TSA 의 cD1 에서는 일부 임펄스만 임계값을 넘고 (고장 첨도 4.0), 원 신호에서는 어느 대역에서도 고장이 정상과 구별되지 않는다. TSA 의 cA3 첨도 (31.6 · 20.0) 가 큰 것은 결함 때문이 아니라 신호 끝의 경계 효과다 (cA3 패널 오른쪽 끝).

---

## 실습 15. Wavelet Denoising — Raw vs TSA vs DIFF

detail 계수에서 threshold 이하(노이즈)를 잘라내고 역변환한다 (Donoho & Johnstone, 1994):

1. `wavedec` 분해
2. 레벨별 노이즈 추정 $\sigma_j = \mathrm{MAD}(cD_j)/0.6745$ — MAD 는 임펄스(outlier)에 강건하며, 0.6745 는 가우시안에서 $\mathrm{MAD} = 0.6745\,\sigma$ 인 환산 상수
3. universal threshold $\lambda_j = \sigma_j\sqrt{2\ln N}$ 로 detail 계수 hard thresholding
4. `waverec` 재구성

In [ ]:
def wavelet_denoise(x, wavelet='db4', level=3):
    """레벨별 universal threshold (hard) denoising."""
    coeffs = pywt.wavedec(x, wavelet, level=level)
    thr = wavelet_threshold(coeffs, len(x))
    coeffs = [coeffs[0]] + [pywt.threshold(c, t, mode='hard') for c, t in zip(coeffs[1:], thr)]
    return pywt.waverec(coeffs, wavelet)[:len(x)], thr

den_results = {}
fig, axes = plt.subplots(2, 3, figsize=(18, 6), sharex=True)
for j, (name, sn, sf) in enumerate([('Raw', raw_n_seg, raw_f_seg),
                                    ('TSA', v_TSA_n, v_TSA_f),
                                    ('DIFF', v_dif_n, v_dif_f)]):
    den_n, _ = wavelet_denoise(sn)
    den_f, thr = wavelet_denoise(sf)
    den_results[name] = (sn, sf, den_n, den_f)
    for i, (a, b, tag) in enumerate([(sf, sn, 'original'), (den_f, den_n, f'denoised (thr_cD1={thr[-1]:.2f})')]):
        ax = axes[i, j]
        for p in idx_pulse:
            ax.axvline(p, color='gray', ls='--', lw=0.6, alpha=0.5)
        ax.plot(a, color='C3', lw=0.8, label='Fault')
        ax.plot(b, color='C0', alpha=0.6, lw=0.8, label='Normal')
        ax.set_ylabel('Amplitude'); ax.legend()
        ax.set_title(f'{name} — {tag}')
for j in range(3):
    axes[-1, j].set_xlabel('Samples (dashed: impulse positions)')
fig.tight_layout(); plt.show()

print(f'{"Signal":8s}{"N":>8s}{"N dn":>8s}{"F":>8s}{"F dn":>8s}   (kurtosis before/after)')
for name, (sn, sf, dn, df_) in den_results.items():
    print(f'{name:8s}{stats.kurtosis(sn, fisher=False):8.2f}{stats.kurtosis(dn, fisher=False):8.2f}'
          f'{stats.kurtosis(sf, fisher=False):8.2f}{stats.kurtosis(df_, fisher=False):8.2f}')

**관찰.**
- 차이 신호: 임계처리 뒤 다섯 임펄스가 모두 남고 그 사이 노이즈가 거의 0 이 된다. 고장 첨도는 19.4 → 58.3 으로 오르고 정상은 2.8 → 2.4 로 오히려 내려가 두 상태가 크게 갈린다.
- TSA: 메쉬 성분이 cD2 · cD3 을 지배해 레벨별 임계값이 계수보다 커지고, 메쉬와 함께 임펄스 대부분이 지워진다. 고장 첨도 61.4 는 240 샘플 근처의 임펄스 하나만 남아서 생긴 값이고 정상 첨도도 11.3 으로 올라가므로, 이 숫자만으로 결함을 판정하기는 어렵다.
- 원 신호: 노이즈가 커서 임계값 (cD1 에서 1.94) 이 임펄스 진폭보다 크고, 남는 것은 임펄스 위치와 무관한 저주파 출렁임뿐이다.

---

## 정리

- 회전 속도 변동 신호의 일반 FFT는 스펙트럼이 퍼져 해석이 어렵다.
- 각도 기반 재샘플링으로 시간축 → 각도축 변환하면 **Order 스펙트럼**에서 첨예한 피크를 얻는다.
- 정수 회전수를 맞추는 방법: **Zero-padding** 또는 **Trigger (trig_rot=1)**.
- **TSA**: 회전주기로 잘라 각도에 맞춰 평균 → 비동기 노이즈가 $1/\sqrt{N}$ 로 줄고 (220 회전 평균에서 약 16 배) 주기적 이상치가 드러남.
- **잔차·차이 신호 + 포락선 스펙트럼**: GMF·사이드밴드를 order-domain 에서 제거하고, 포락선 스펙트럼의 1x 배수 comb 로 임펄스 반복 주기를 확인.
- **기어 상태 피처**: TSA·잔차·차이 신호에서 FM0·SER·NA4·FM4·M6A·M8A·ER 스칼라 지표를 계산 → 정상/고장을 수치 하나로 비교, 고차 모멘트일수록 임펄스성 결함에 민감.
- **Cepstrum·EEMD·웨이블릿**: 같은 분석도 입력 신호에 따라 결과가 달라진다. cepstrum 은 TSA, 분해·denoising 은 DIFF 에서 결함이 가장 잘 드러나고, Raw 에 바로 적용하면 묻히거나 오탐이 생긴다.